In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("melikechan/cifar100")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cifar100' dataset.
Path to dataset files: /kaggle/input/cifar100


In [10]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

In [11]:
train_data = datasets.ImageFolder('/kaggle/input/cifar100/cifar100/train', transform=transform)
test_data = datasets.ImageFolder('/kaggle/input/cifar100/cifar100/test', transform=transform)

In [12]:
train = DataLoader(train_data, batch_size=32, shuffle=True)
test = DataLoader(test_data, batch_size=32, shuffle=False)


In [14]:
len(train_data.classes)

100

In [21]:
class CheckImage(nn.Module):
  def __init__(self):
    super().__init__()
    self.first = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )

    self.second = nn.Sequential(
        nn.Flatten(),
        nn.Linear(128 * 8 * 8, 256),
        nn.ReLU(),
        nn.Linear(256, 100)
    )

  def forward(self, x):
    x = self.first(x)
    x = self.second(x)
    return x

In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [23]:
model = CheckImage().to(device)

In [24]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [34]:
for epoch in range(5):
  model.train()
  total_loss = 0
  for x_batch, y_batch in train:
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)

    y_pred = model(x_batch)
    loss = loss_fn(y_pred, y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  print(f'Epoch: {epoch+1}, Loss: {total_loss}')

Epoch: 1, Loss: 593.7398169897497
Epoch: 2, Loss: 557.0760807087645
Epoch: 3, Loss: 547.5169136375189
Epoch: 4, Loss: 533.1588913910091
Epoch: 5, Loss: 484.76344949472696


In [35]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
  for x_batch, y_batch in test:
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)


    y_pred = model(x_batch)
    predicted = torch.argmax(y_pred, dim=1)

    total += y_batch.size(0)
    correct += (predicted == y_batch).sum().item()

accuracy = 100 * correct/ total

print(f'{round(accuracy, 2)}%')




93.99%


In [36]:
torch.save(model.state_dict(), 'model_cifar.pth')